In [ ]:
# 2x2 LER panel for repetition experiments (MWPM / UF / UF_list)
# ISCA-style: larger markers/fonts, thicker lines, compact layout
import os
import glob
import json
from typing import Dict, Any
import numpy as np
import matplotlib.pyplot as plt

# Global style close to compose_plots.ipynb
plt.rcParams.update({
    'font.size': 24,
    'axes.labelsize': 24,
    'axes.titlesize': 24,
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'legend.fontsize': 22,
    'lines.linewidth': 3.2,
    'lines.markersize': 16,
    'grid.alpha': 0.35,
})

palette = {
    'mwpm': '#1f77b4',      # blue
    'uf': '#ff7f0e',        # orange
    'uf_list': '#2ca02c',   # green
}

marker_map = {'mwpm': 'o', 'uf': 's', 'uf_list': '^'}
label_map = {'mwpm': 'MWPM-based Decoder', 'uf': 'UF-based Decoder', 'uf_list': 'Ours'}

PLOT_AS_SCATTER = True
SCATTER_SIZE = 160
MARKER_EDGE_WIDTH = 1.8
X_JITTER = 1e-6

# Root where AccExpRepetition.py writes outputs
ROOT_DIR = "rebuttal_outputs/repetition_code"
L_TARGETS = [5, 7]   # 1x2
OUT_DIR = "figplot/repetition_code"
os.makedirs(OUT_DIR, exist_ok=True)

def _load_latest_acc_json_for_L(root: str, L: int) -> Dict[str, Any]:
    # Try acc_ler format first (from 1_AccExpRepetition.py)
    candidates = glob.glob(os.path.join(root, f"**/acc_ler_L{L}_list*.json"), recursive=True)
    if candidates:
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
        with open(candidates[0], "r") as f:
            return json.load(f)
    # Fallback: try plots_data format (from run_ablation_pipeline.py)
    candidates = glob.glob(os.path.join(root, f"**/plots_data_L{L}_*.json"), recursive=True)
    if candidates:
        candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
        with open(candidates[0], "r") as f:
            d = json.load(f)
        # Map keys: uf_peel_votemax -> uf_list
        ler = d.get('ler', {})
        mapped_ler = {
            'mwpm': ler.get('mwpm', []),
            'uf': ler.get('uf', []),
            'uf_list': ler.get('uf_peel_votemax', ler.get('uf_efficient_votemax', [])),
        }
        return {'ps': d.get('ps', []), 'ler': mapped_ler}
    return {}

by_L: Dict[int, Dict[str, Any]] = {}
for L in L_TARGETS:
    d = _load_latest_acc_json_for_L(ROOT_DIR, L)
    if d:
        by_L[L] = d

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

for i, L in enumerate(L_TARGETS):
    ax = axes[i]
    data = by_L.get(L)
    if not data:
        ax.set_title(f"Code Distance={L} (N/A)")
        ax.axis("off")
        continue

    ps = data.get("ps", [])
    ler = data.get("ler", {})

    for key in ("mwpm", "uf", "uf_list"):
        y = ler.get(key, [])
        n = min(len(ps), len(y))
        if n <= 0:
            continue

        xp = np.asarray(ps[:n], dtype=float)
        yp = np.asarray(y[:n], dtype=float)

        if PLOT_AS_SCATTER:
            if key == 'mwpm':
                xs = xp - X_JITTER
            elif key == 'uf_list':
                xs = xp + X_JITTER
            else:
                xs = xp
            ax.scatter(
                xs, yp,
                s=SCATTER_SIZE,
                marker=marker_map[key],
                color=palette[key],
                edgecolor='black',
                linewidths=MARKER_EDGE_WIDTH,
                label=label_map[key],
                zorder=3,
            )
            ax.plot(xp, yp, linestyle='-', color=palette[key], linewidth=2.8, zorder=2)
        else:
            ax.plot(
                xp, yp,
                marker=marker_map[key], linestyle='-',
                color=palette[key], linewidth=2.8,
                label=label_map[key],
            )

    ax.set_yscale("log")
    ax.grid(True, axis='y', linestyle='--', linewidth=1.8)
    for gl in ax.get_ygridlines():
        gl.set_linestyle((0, (6, 6)))
        gl.set_linewidth(1.8)
        gl.set_color('#9a9a9a')
        gl.set_alpha(0.65)

    ax.set_title(f"Code Distance {L}", fontsize=24)
    ax.set_xlabel("Physical Error Rate", fontsize=24)
    if i == 0:
        ax.set_ylabel("Logical Error Rate", fontsize=24)

# Shared legend at top with framed box
handles_dict = {}
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for handle, label in zip(h, l):
        if label and label not in handles_dict:
            handles_dict[label] = handle

if handles_dict:
    desired_order = ['MWPM-based Decoder', 'UF-based Decoder', 'Ours']
    ordered_labels = [lbl for lbl in desired_order if lbl in handles_dict]
    ordered_handles = [handles_dict[lbl] for lbl in ordered_labels]
    leg = fig.legend(
        ordered_handles,
        ordered_labels,
        loc='upper center',
        ncol=len(ordered_labels),
        frameon=True,
        bbox_to_anchor=(0.5, 1.00),
    )
    frame = leg.get_frame()
    frame.set_edgecolor('black')
    frame.set_linewidth(1.8)
    frame.set_facecolor('white')
    frame.set_alpha(1.0)

fig.tight_layout(rect=[0, 0.01, 1, 0.94])
save_path = os.path.join(OUT_DIR, "combined_ler_mwpm_uf_uflist_1x2.pdf")
fig.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.02, format='pdf')
plt.show()
print(f"Saved: {save_path}")
